# 04 · Tech Wins & Outcomes
Tracks use case tech wins, UC wins, and go-lives attributed to the specialist team.
Joins to `MDM.MDM_INTERFACES.DIM_USE_CASE` and `USE_CASE_ATTRIBUTION`.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
fy_year      = 2026   # Snowflake FY (starts Feb 1)
quarter      = "Q2"   # Q1=Feb-Apr, Q2=May-Jul, Q3=Aug-Oct, Q4=Nov-Jan
# ─────────────────────────────────────────────────────────
quarter_dates = {
    "Q1": (f"{fy_year-1}-02-01", f"{fy_year-1}-04-30"),
    "Q2": (f"{fy_year-1}-05-01", f"{fy_year-1}-07-31"),
    "Q3": (f"{fy_year-1}-08-01", f"{fy_year-1}-10-31"),
    "Q4": (f"{fy_year-1}-11-01", f"{fy_year}-01-31"),
}
# For FY26 Q2 -> May 1 2026 - Jul 31 2026
qs, qe = "2026-05-01", str(today)  # cap at today
fy_start_str = "2026-02-01"
print(f"FY{fy_year} {quarter}  |  {qs} → {qe}")


## Tech Win & Go-Live KPIs (QTD)

In [ ]:
%%sql -r tw_kpi
WITH attributed AS (
    SELECT DISTINCT a.USE_CASE_ID, a.USER_ID, a.TEAM_ROLE,
           d.USE_CASE_NAME, d.ACCOUNT_NAME, d.IS_TECH_WON,
           d.IS_WON, d.IS_DEPLOYED, d.IS_LOST,
           d.DECISION_DATE, d.ACTUAL_USE_CASE_DEPLOYMENT_DATE,
           d.USE_CASE_EACV, d.USE_CASE_STAGE, d.STAGE_NUMBER
    FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
    JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
    WHERE a.USER_ID IN ('{{team_ids_str}}')
      AND a.TEAM_ROLE IS NOT NULL
)
SELECT
    SUM(IFF(IS_TECH_WON AND DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}',1,0))              AS tech_wins_qtd,
    SUM(IFF(IS_WON AND DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}',1,0))                   AS uc_wins_qtd,
    SUM(IFF(IS_DEPLOYED AND ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{qs}}' AND '{{qe}}',1,0)) AS go_lives_qtd,
    ROUND(SUM(IFF(IS_TECH_WON AND DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}', COALESCE(USE_CASE_EACV,0),0))/1e6,2) AS tw_eacv_m,
    ROUND(SUM(IFF(IS_DEPLOYED AND ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{qs}}' AND '{{qe}}', COALESCE(USE_CASE_EACV,0),0))/1e6,2) AS gl_eacv_m,
    SUM(IFF(STAGE_NUMBER BETWEEN 1 AND 6,1,0))                                              AS active_pipeline_ucs
FROM attributed

In [ ]:
tw_kpi = tw_kpi.to_pandas() if hasattr(tw_kpi, "to_pandas") else tw_kpi
r = tw_kpi.iloc[0]
fig, axes = plt.subplots(1,3, figsize=(13,3))
for ax in axes: clean(ax)
kpis = [
    (int(r.get("TECH_WINS_QTD",0) or 0), "Tech Wins QTD", SF_BLUE),
    (int(r.get("GO_LIVES_QTD",0) or 0),  "Go-Lives QTD",  SF_GREEN),
    (int(r.get("ACTIVE_PIPELINE_UCS",0) or 0), "Active Pipeline UCs", SF_TEAL),
]
for ax, (val, title, color) in zip(axes, kpis):
    ax.text(0.5, 0.55, str(val), ha="center", va="center", fontsize=42, fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.2, title, ha="center", va="center", fontsize=12, color=SF_DARK, transform=ax.transAxes)
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis("off")
plt.suptitle(f"FY{fy_year} {quarter} Team KPIs  |  eACV: Tech Wins ${r.get('TW_EACV_M',0):.1f}M  |  Go-Lives ${r.get('GL_EACV_M',0):.1f}M", color=SF_DARK, fontsize=11)
plt.tight_layout(); plt.show()

## Recent Tech Wins & Go-Lives

In [ ]:
%%sql -r recent_wins
SELECT DISTINCT
    d.ACCOUNT_NAME, d.USE_CASE_NAME,
    CASE WHEN d.IS_TECH_WON AND d.DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}' THEN 'Tech Win'
         WHEN d.IS_DEPLOYED AND d.ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{qs}}' AND '{{qe}}' THEN 'Go Live'
         WHEN d.IS_WON AND d.DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}' THEN 'UC Win'
    END                                     AS milestone,
    COALESCE(d.DECISION_DATE, d.ACTUAL_USE_CASE_DEPLOYMENT_DATE) AS milestone_date,
    d.USE_CASE_EACV,
    d.USE_CASE_STAGE,
    n.ACTIVITY_SE_NAME                      AS specialist
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
JOIN SALES.SE_REPORTING.DIM_SE_ACTIVITY n ON a.USER_ID = n.ACTIVITY_SE_ID
WHERE a.USER_ID IN ('{{team_ids_str}}')
  AND a.TEAM_ROLE IS NOT NULL
  AND (
    (d.IS_TECH_WON AND d.DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}')
    OR (d.IS_DEPLOYED AND d.ACTUAL_USE_CASE_DEPLOYMENT_DATE BETWEEN '{{qs}}' AND '{{qe}}')
    OR (d.IS_WON AND d.DECISION_DATE BETWEEN '{{qs}}' AND '{{qe}}')
  )
ORDER BY milestone_date DESC LIMIT 50

In [ ]:
recent_wins = recent_wins.to_pandas() if hasattr(recent_wins, "to_pandas") else recent_wins
def win_color(r):
    m = str(r.get("MILESTONE",""))
    return "#d4edda" if m=="Go Live" else "#cce5ff" if m=="Tech Win" else "#fff3cd"
html_table(recent_wins, row_color_fn=win_color)

## Active Pipeline — Use Case Stage Distribution

In [ ]:
%%sql -r pipeline_stages
SELECT
    d.USE_CASE_STAGE,
    d.STAGE_NUMBER,
    COUNT(DISTINCT d.USE_CASE_ID)            AS use_cases,
    ROUND(SUM(d.USE_CASE_EACV)/1e6, 2)      AS eacv_m
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION a
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON a.USE_CASE_ID = d.USE_CASE_ID
WHERE a.USER_ID IN ('{{team_ids_str}}')
  AND a.TEAM_ROLE IS NOT NULL
  AND d.IS_LOST = FALSE
  AND d.STAGE_NUMBER BETWEEN 1 AND 6
GROUP BY 1,2 ORDER BY 2

In [ ]:
pipeline_stages = pipeline_stages.to_pandas() if hasattr(pipeline_stages, "to_pandas") else pipeline_stages
df = pipeline_stages.sort_values("STAGE_NUMBER")
fig, ax = plt.subplots(figsize=(12,4)); clean(ax)
colors = [SF_BLUE, SF_TEAL, SF_GREEN, SF_ORANGE, "#9B59B6", "#E74C3C"]
bars = ax.bar(df["USE_CASE_STAGE"], df["USE_CASES"], color=colors[:len(df)], alpha=0.85)
ax2 = ax.twinx()
ax2.plot(df["USE_CASE_STAGE"], df["EACV_M"], color=SF_DARK, marker="o", linewidth=2, label="eACV ($M)")
for bar, val in zip(bars, df["USE_CASES"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(int(val)),
            ha="center", va="bottom", fontsize=10, color=SF_DARK)
ax.set_title("Active Pipeline by Stage", fontsize=13, color=SF_DARK)
ax.set_ylabel("Use Cases"); ax2.set_ylabel("eACV ($M)", color=SF_DARK)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## Time-to-Tech-Win Analysis (by Engagement)

In [ ]:
%%sql -r ttw
SELECT
    d.ACCOUNT_NAME,
    d.USE_CASE_NAME,
    MIN(a_hist.ACTIVITY_DATE)       AS first_engagement,
    d.DECISION_DATE                 AS tech_win_date,
    DATEDIFF('day', MIN(a_hist.ACTIVITY_DATE), d.DECISION_DATE) AS days_to_tw,
    d.USE_CASE_EACV
FROM SALES.SE_REPORTING.USE_CASE_ATTRIBUTION attr
JOIN MDM.MDM_INTERFACES.DIM_USE_CASE d ON attr.USE_CASE_ID = d.USE_CASE_ID
JOIN SALES.SE_REPORTING.DIM_SE_ACTIVITY a_hist
    ON attr.USER_ID = a_hist.ACTIVITY_SE_ID
    AND a_hist.ACCOUNT_ID = d.ACCOUNT_ID
WHERE attr.USER_ID IN ('{{team_ids_str}}')
  AND d.IS_TECH_WON = TRUE
  AND d.DECISION_DATE >= '{{fy_start_str}}'
GROUP BY 1,2,4,6
HAVING days_to_tw >= 0
ORDER BY days_to_tw

In [ ]:
ttw = ttw.to_pandas() if hasattr(ttw, "to_pandas") else ttw
if (len(ttw.to_pandas()) if hasattr(ttw, "to_pandas") else len(ttw)) > 0:
    fig, ax = plt.subplots(figsize=(12,4)); clean(ax)
    ax.barh(ttw["USE_CASE_NAME"].str[:40], ttw["DAYS_TO_TW"], color=SF_BLUE, alpha=0.85)
    ax.set_xlabel("Days from First Engagement to Tech Win")
    ax.set_title(f"Time to Tech Win (Avg: {ttw['DAYS_TO_TW'].mean():.0f} days)", color=SF_DARK)
    plt.tight_layout(); plt.show()
    print(f"Fastest: {ttw['DAYS_TO_TW'].min()} days | Median: {ttw['DAYS_TO_TW'].median():.0f} days | Slowest: {ttw['DAYS_TO_TW'].max()} days")
else:
    print("No tech wins with traceable engagement history in this period.")